In [2]:
import rasterio
from rasterio.windows import Window
import numpy as np

# 1. Open the file header without loading the pixels into RAM
with rasterio.open('slope.tif') as src:
    print(f"Full image dimensions: {src.width}x{src.height}")
    meta = src.meta.copy()
    
    # 2. Define a Window to read a subset
    # Format: Window(col_off, row_off, width, height)
    # Let's read a 10,000 x 10,000 chunk from the middle/center-bottom where Faustini is.
    # You can tweak these offsets to target your specific coordinates!
    col_start = int(src.width * 0.4)   # starting at 40% of width
    row_start = int(src.height * 0.6)  # starting at 60% of height
    chunk_size = 10000                 # 10k x 10k pixels (~400 MB in RAM, super safe)
    
    window = Window(col_start, row_start, chunk_size, chunk_size)
    
    # 3. Read ONLY that window
    slope_matrix = src.read(1, window=window)
    
    # 4. Update metadata to match our new smaller chunk
    meta.update({
        'height': chunk_size,
        'width': chunk_size,
        'transform': rasterio.windows.transform(window, src.transform)
    })

# Now do the exact same window read for aspect so they align perfectly
with rasterio.open('aspect.tif') as src:
    aspect_matrix = src.read(1, window=window)

print("Successfully loaded chunks! Array shape:", slope_matrix.shape)

Full image dimensions: 95018x79873
Successfully loaded chunks! Array shape: (10000, 10000)


In [3]:
from scipy.ndimage import uniform_filter

# Calculate local standard deviation of the slope
mean_slope = uniform_filter(slope_matrix, size=5)
sq_mean_slope = uniform_filter(slope_matrix**2, size=5)
slope_std = np.sqrt(np.maximum(sq_mean_slope - mean_slope**2, 0))

In [4]:
# Create Boolean masks based on mission criteria
safe_slope_mask = (slope_matrix >= 0) & (slope_matrix <= 10)
safe_roughness_mask = slope_std < 1.5 
good_sunlight_mask = (aspect_matrix > 135) & (aspect_matrix < 225) # Adjust degrees based on ephemeris data

In [5]:
# Combine all criteria. A pixel is only 1 (Safe) if it satisfies ALL conditions.
feasible_landing_zones = safe_slope_mask & safe_roughness_mask & good_sunlight_mask

In [10]:
# Create a fresh metadata dictionary completely from scratch
clean_meta = {
    'driver': 'GTiff',
    'dtype': 'uint8',
    'count': 1,
    'height': feasible_landing_zones.shape[0],
    'width': feasible_landing_zones.shape[1],
    'crs': meta.get('crs'),          # Keeps your Moon coordinate reference system
    'transform': meta.get('transform') # Keeps your window positioning mapping
}

with rasterio.open('feasible_landing_zones.tiff', 'w', **clean_meta) as dst:
    dst.write(feasible_landing_zones.astype(np.uint8), 1)

print("File saved successfully!")

File saved successfully!


In [11]:
print("Total number of pixels:", feasible_landing_zones.size)
print("Number of safe landing pixels found (Value = 1):", np.sum(feasible_landing_zones == 1))

Total number of pixels: 100000000
Number of safe landing pixels found (Value = 1): 76606


In [ ]:
import rasterio
import numpy as np
from scipy.ndimage import label, center_of_mass

def find_top_landing_sites(binary_tif_path, num_sites=5):
    # 1. Load the binary mask you generated
    with rasterio.open(binary_tif_path) as src:
        mask = src.read(1)
        transform = src.transform
        crs = src.crs

    print("Analyzing pixel adjacencies... (this may take a moment)")
    
    # 2. Label contiguous groups of 1s
    # structure=np.ones((3,3)) ensures diagonal connections (8-connectivity) are counted
    labeled_array, num_features = label(mask, structure=np.ones((3,3)))
    print(f"Found a total of {num_features} distinct contiguous patches.")

    if num_features == 0:
        print("No landing patches found! Try relaxing your thresholds first.")
        return

    # 3. Count the size (number of pixels) of each labeled cluster
    # Index 0 is the background (unsafe pixels), so we ignore it
    cluster_sizes = np.bincount(labeled_array.ravel())
    cluster_sizes[0] = 0  # Ignore background cluster

    # 4. Get the indices of the top N largest clusters in descending order
    top_cluster_indices = np.argsort(cluster_sizes)[::-1][:num_sites]

    print("\n--- TOP 5 LANDING SITES FOUND ---")
    
    landing_sites_data = []
    
    for rank, cluster_id in enumerate(top_cluster_indices, 1):
        size_in_pixels = cluster_sizes[cluster_id]
        if size_in_pixels == 0:
            continue
            
        # 5. Find the center of mass (centroid pixel location) of this cluster
        # Note: center_of_mass returns (row, col)
        row_centroid, col_centroid = center_of_mass(mask, labeled_array, cluster_id)
        
        # 6. Convert the pixel index center to actual Lunar Map Coordinates (X, Y)
        map_x, map_y = transform * (col_centroid, row_centroid)
        
        print(f"Rank {rank}: Cluster ID {cluster_id}")
        print(f"   - Size: {size_in_pixels} contiguous safe pixels")
        print(f"   - Pixel Center: (Row: {int(row_centroid)}, Col: {int(col_centroid)})")
        print(f"   - Lunar Coordinates (X, Y): ({map_x:.2f}, {map_y:.2f})")
        
        landing_sites_data.append({
            'rank': rank,
            'size': size_in_pixels,
            'x': map_x,
            'y': map_y
        })
        
    return landing_sites_data

# Run the function on your file
top_sites = find_top_landing_sites('feasible_landing_zones.tiff', num_sites=5)

In [12]:
import rasterio
import numpy as np
from scipy.ndimage import label

def export_top_5_raster(binary_tif_path, output_tif_path, num_sites=5):
    # 1. Load your original binary mask layer
    with rasterio.open(binary_tif_path) as src:
        mask = src.read(1)
        original_meta = src.meta.copy()

    print("Analyzing pixel adjacencies and grouping clusters...")
    
    # 2. Label contiguous groups using 8-connectivity (includes diagonals)
    # This turns your 1s into unique cluster numbers (e.g., Cluster 12, Cluster 45...)
    labeled_array, num_features = label(mask, structure=np.ones((3,3)))
    
    # 3. Calculate the size (pixel count) of each distinct patch
    cluster_sizes = np.bincount(labeled_array.ravel())
    cluster_sizes[0] = 0  # Force background pixel count to 0
    
    # 4. Extract the IDs of the top 5 largest clusters in descending order
    top_cluster_ids = np.argsort(cluster_sizes)[::-1][:num_sites]
    
    # 5. Create a clean, blank array of zeros for our final top sites map
    top_sites_mask = np.zeros_like(labeled_array, dtype=np.uint8)
    
    print("\nExtracting and saving top landing patches:")
    # 6. Burn the top clusters into our new array
    # They are assigned values 1, 2, 3, 4, 5 corresponding directly to their size rank!
    for rank, cluster_id in enumerate(top_cluster_ids, 1):
        if cluster_sizes[cluster_id] > 0:
            top_sites_mask[labeled_array == cluster_id] = rank
            print(f"   * Rank {rank} (Cluster ID {cluster_id}) -> Size: {cluster_sizes[cluster_id]} contiguous pixels")

    # 7. Configure clean metadata explicitly to avoid any writer/driver conflicts
    clean_meta = {
        'driver': 'GTiff',
        'dtype': 'uint8',
        'count': 1,
        'height': top_sites_mask.shape[0],
        'width': top_sites_mask.shape[1],
        'crs': original_meta.get('crs'),          # Retains Lunar Map CRS
        'transform': original_meta.get('transform')  # Retains window alignment coordinates
    }

    # 8. Write the final array out as a GeoTIFF
    with rasterio.open(output_tif_path, 'w', **clean_meta) as dst:
        dst.write(top_sites_mask, 1)
        
    print(f"\nSuccessfully generated file: '{output_tif_path}'")

# Execute the script
export_top_5_raster('feasible_landing_zones.tiff', 'top_5_landing_sites.tiff')

Analyzing pixel adjacencies and grouping clusters...

Extracting and saving top landing patches:
   * Rank 1 (Cluster ID 46) -> Size: 944 contiguous pixels
   * Rank 2 (Cluster ID 6246) -> Size: 533 contiguous pixels
   * Rank 3 (Cluster ID 11265) -> Size: 515 contiguous pixels
   * Rank 4 (Cluster ID 6266) -> Size: 347 contiguous pixels
   * Rank 5 (Cluster ID 13396) -> Size: 332 contiguous pixels

Successfully generated file: 'top_5_landing_sites.tiff'
